In [1]:
import joblib
import pandas as pd
import numpy as np
import json
import os

In [ ]:
# 4. Pipeline Export für Backend

# Pfade definieren
MODEL_DIR = '../ml/model'
PIPELINE_PATH = os.path.join(MODEL_DIR, 'diabetes_pipeline.pkl') # Die Pipeline aus Teil 2
EXPLAINER_PATH = os.path.join(MODEL_DIR, 'shap_explainer.pkl')
FEATURES_PATH = os.path.join(MODEL_DIR, 'selected_features.json')
DATA_PATH = '../data/diabetes_binary_health_indicators_BRFSS2015.csv'

print("Lade Komponenten für den Export...")

# 1. Pipeline und Explainer laden
if not os.path.exists(PIPELINE_PATH):
    print(f"Fehler: Pipeline unter {PIPELINE_PATH} nicht gefunden!")
    exit()

pipeline = joblib.load(PIPELINE_PATH)
explainer = joblib.load(EXPLAINER_PATH)

with open(FEATURES_PATH, 'r') as f:
    selected_features = json.load(f)

# 2. Statistiken vorab berechnen
df = pd.read_csv(DATA_PATH)
feature_stats = {}
for feature in selected_features:
    feature_stats[feature] = {
        'min': float(df[feature].min()),
        'max': float(df[feature].max()),
        'median': float(df[feature].median()),
        'mean': float(df[feature].mean())
    }

def get_risk_assessment(input_data, pipeline, explainer, feature_stats, features_list):
    """
    Berechnet Risiko und SHAP-Einfluss basierend auf der Pipeline.
    input_data: Dictionary mit User-Eingaben
    """
    # 1. Eingabewerte vorbereiten & fehlende mit Median füllen
    final_input = []
    for f in features_list:
        val = input_data.get(f, feature_stats[f]['median'])
        final_input.append(float(val))
    
    # 2. In DataFrame umwandeln
    input_df = pd.DataFrame([final_input], columns=features_list)
    
    # 3. Vorhersage über die Pipeline
    risk_proba = pipeline.predict_proba(input_df)[0][1]
    
    # 4. SHAP Werte berechnen
    scaler = pipeline.named_steps['scaler']
    rf_model = pipeline.named_steps['clf']
    
    input_scaled = scaler.transform(input_df)
    shap_values = explainer.shap_values(input_scaled)
    
    # SHAP-Format korrigieren
    if isinstance(shap_values, list):
        shap_vals_class1 = shap_values[1][0]
    elif len(shap_values.shape) == 3:
        shap_vals_class1 = shap_values[0, :, 1]
    else:
        shap_vals_class1 = shap_values[0]

    # Feature Impacts zusammenbauen
    impacts = []
    for i, feature in enumerate(features_list):
        impacts.append({
            'feature': feature,
            'impact': round(float(shap_vals_class1[i]), 4),
            'value': final_input[i]
        })
    
    # Sortieren für Top-Faktoren
    impacts_sorted = sorted(impacts, key=lambda x: x['impact'], reverse=True)
    
    return {
        'risk_percent': round(risk_proba * 100, 1),
        'risk_level': 'Hoch' if risk_proba > 0.5 else 'Mittel' if risk_proba > 0.2 else 'Niedrig',
        'top_drivers': impacts_sorted[:3],     # Risiko erhöhen
        'top_protectors': impacts_sorted[-3:]  # Risiko senken
    }

print("\nTeste Risiko-Einstufung...")
example_user = {'BMI': 38, 'HighBP': 1, 'Age': 11} # Rest wird Median
test_result = get_risk_assessment(example_user, pipeline, explainer, feature_stats, selected_features)
print(f"Ergebnis: {test_result['risk_percent']}% Risiko ({test_result['risk_level']})")

# Beschreibungen für die UI
feature_descriptions = {
    'HighBP': 'Bluthochdruck',
    'HighChol': 'Hoher Cholesterinspiegel',
    'BMI': 'Body-Mass-Index',
    'Smoker': 'Raucherstatus',
    'Stroke': 'Früherer Schlaganfall',
    'HeartDiseaseorAttack': 'Herzprobleme',
    'PhysActivity': 'Körperliche Aktivität',
    'Fruits': 'Täglicher Obstkonsum',
    'Veggies': 'Täglicher Gemüsekonsum',
    'HvyAlcoholConsump': 'Starker Alkoholkonsum',
    'GenHlth': 'Allg. Gesundheitszustand (1-5)',
    'MentHlth': 'Tage mit psych. Problemen / Monat',
    'PhysHlth': 'Tage mit phys. Problemen / Monat',
    'Age': 'Altersgruppe (1-13)',
    'Education': 'Bildungsgrad',
    'Income': 'Einkommensklasse'
}

frontend_config = {
    'features': selected_features,
    'feature_info': {f: {
        'label': feature_descriptions.get(f, f),
        'min': feature_stats[f]['min'],
        'max': feature_stats[f]['max'],
        'default': feature_stats[f]['median']
    } for f in selected_features}
}

# Speichern
with open(os.path.join(MODEL_DIR, 'frontend_config.json'), 'w', encoding='utf-8') as f:
    json.dump(frontend_config, f, indent=2, ensure_ascii=False)

joblib.dump(pipeline, os.path.join(MODEL_DIR, 'final_model_pipeline.joblib'))

print("\nExport erfolgreich abgeschlossen.")

Lade Komponenten für den Export...

Teste Risiko-Einstufung...
Ergebnis: 59.2% Risiko (Hoch)

Export erfolgreich abgeschlossen.
